# Preprocessing

### Introduction

This section contains the functions that will be applied in the cross validation as well as some general DETERMINISTIC corrections, performed before and outside the folds.

- This file contains all the transformations that can be performed outside and before the fols of our cross validation algorithm.
- This correspond to the pre processing functions that will not induce data leackage.
- We decided to perform some of the safe operations outside the folds in order to make this project run more efficiently, by avoiding to reapply this step for all folds, we just do it once. 

## 1. Imports and dataset 

In [1]:
import pandas as pd
import unicodedata
import re
from typing import Dict, Any
import numpy as np


In [40]:
full_train_dataset = pd.read_csv('../../project_data/train.csv')

## 2. Pre-processing rationale

- The main idea behind our data preprocessing pipeline is to mimic the standard fit and transform logic.
- We explicitly split each correction (previously studied in notebook xxx) into a fit function and a transform function. The __fit__ functions compute all statistics and imputations using only the __training data__ of each fold (e.g.: medians, means, modes, mappings, etc), while the __transform__ functions apply purely deterministic operations (such as clipping, taking absolute values, type casting, etc) and/or replace values using the statistics learned during fit (for example, imputing missing values with the previously computed median) on __both the training and validation sets__. 
- This design was adopted to minimise data leakage, which is a common and serious issue in machine learning. By enforcing a clear separation between fitting and transforming, we ensure that no information from the validation folds influences the learned imputations or corrections. We also did some improvements over our first delivery, where some functions were redundant or confusing.
- Additionally, some functions are applied once before cross-validation to the full provided training set (which will later be split into folds). These are strictly deterministic transformations that do not require any statistics from the data (for instance, basic string normalisation, uppercasing, stripping spaces, fixed pattern replacements). Because they do not depend on the distribution of the data, they don't introduce data leakage. Applying them once globally or separately per fold would cause identical results, but doing them upfront is more computationally efficient, since we avoid recomputing the same deterministic operations for every fold. 

### 2.1. Pseudo-code

Tranformations applied here:
- Year: minimum year and max year;
- Percentages within 0-100%;
- Dropping has damage and carID;
- 

## Missing values replacement for categorical columns

## the upper, strip, erase weird characters, 

This function:
- keeps letters and digit;
- transforms full string to upper case;
- removes spaces in the middle, beggining and end; (might allow spaces in the middle)
- removes symbols and ponctuation; (might allow keeping -)
- removes acentos of the vogals.

In [44]:
## replace categorical features with missing values with UNKNOWN

def fill_unknown(series):
    return series.fillna("UNKNOWN")


In [3]:
def basic_string_transformer(word, 
    remove_middle_spaces: bool = True,
    allow_extra_chars: str = ""
    ):

    if pd.isna(word):
        return word
    
    s = str(word)

    # 1st: strip the leading and trailing whitespaces 
    s = s.strip()

    # 2nd: uppercase the string
    s = s.upper()

    # 3rd: removal of accents
    s = unicodedata.normalize("NFD", s)
    s = "".join(ch for ch in s if unicodedata.category(ch) != "Mn")

    # 4th: remove middle spaces if required
    if remove_middle_spaces:
        s = s.replace(" ", "")
    else:
        s = re.sub(r"\s+", " ", s)
        s = s.strip()
    
    # 5th: remove ponctuation/symbols etc
    # basically, by default, only keeps A-Z characters and digits
    # optionally if the user selects extra allowed characters, we keep them too
    allowed = allow_extra_chars or "" 
    # if we want to keep middle spaces, we add space to allowed characters
    if not remove_middle_spaces:
        allowed += " "
    # pattern = f"[^A-Z0-9{re.escape(allowed)}]" we did not use this because treats it as a raw string - possible problem with new line 
    pattern = rf"[^A-Z0-9{re.escape(allowed)}]"
    s = re.sub(pattern, "", s)

    #6th: if string becomes empty or only has spaces, return NaN
    if s.strip() == "":
        return np.nan
    
    return s

In [4]:
def def_string_basic_transformer(
    df: pd.DataFrame,
    column: str,
    remove_middle_spaces: bool = True,
    allow_extra_chars: str = ""
    ) -> pd.DataFrame: #returns a dataframe
    """ 
    Apply basic string normalization to a given column of a DataFrame.

    - Parameters: 
    df : pd.DataFrame
        Input DataFrame.
    column : str
        Name of the column to transform.
    remove_inner_spaces : bool, default True
        If True, remove all spaces (including in the middle).
        If False, keep inner spaces (normalizing multiple spaces to just one).
    allow_extra_chars : str, default ""
        String of extra characters to keep (e.g., "-/" to keep '-' and '/').

    - Returns
    pd.DataFrame
        A copy of df with the specified column transformed.
    """
    
    df_out = df.copy()
    df_out[column] = df_out[column].apply(
        lambda x: basic_string_transformer(
            x,      
            remove_middle_spaces=remove_middle_spaces,
            allow_extra_chars=allow_extra_chars,
        )
    )   
    df_out[column] = df_out[column].astype("string")

    return df_out

In [7]:
valid_brands = ['FORD', 'MERCEDES', 'VW', 'OPEL', 'BMW', 'AUDI', 'TOYOTA', 'SKODA', 'HYUNDAI']

# Fill missing Brand values with 'UNKNOWN'

""" 
# Identify brands in the dataset that are not in the list of valid brands
invalids = sorted(
    [b for b in train['Brand'].unique() if b not in valid_brands],
    key=len
)
"""

" \n# Identify brands in the dataset that are not in the list of valid brands\ninvalids = sorted(\n    [b for b in train['Brand'].unique() if b not in valid_brands],\n    key=len\n)\n"

In [8]:
def correct_invalid_brands_in_df(df, col, valid_brands, invalids):
    # Dictionary to store corrections applied
    corrections = {}
    
    for invalid in invalids:
        if invalid == 'UNKNOWN':
            continue  # Skip UNKNOWN values
        
        # Check if the invalid brand is contained in exactly one valid brand
        matches = [vb for vb in valid_brands if invalid in vb]
        
        if len(matches) == 1:
            valid = matches[0]
            df.loc[df[col] == invalid, col] = valid  # Replace invalid with the valid brand
            corrections[invalid] = valid

    # Identify any remaining invalid values that were not corrected
    remaining_invalids = [
        b for b in df[col].unique() 
        if b not in valid_brands and b not in corrections.keys()
    ]
    
    return df, corrections, remaining_invalids


# Cross val functions

For the transformations to be performed inside the cross validation folds, we followed a fit and transform logic.
- Fit: the fit functions are the ones that are consist of imputations from the dataset, basically calculations such as median, average, etc. This are calculated based on the training dataset.
- Transform: this consist on the transformations that eiter don't rely on calculations or that apply the previously "fit" imputations made (for example, the actual replacement of missing values with the median). 

in total we have 4 catgeorical features, each of them with their own transformations:
1) brand:
    - correct_invalid_brands_in_df -> transformm 
    - correct_ambiguous_brands -> problem: envolves both fitting and transforming. 2 options:
        - aplly outside fold and say its just a semantic correction;
        - split in 2 fit and transform and store the state
    
2) model
    - correct_invalid_models: same problem with split of fitting and transforming

3) fuelType
    - correct_invalid_fueltypes

4) transmission
    - correct_invalid_transmissions 

## 1) Brand

In [9]:
valid_brands 

['FORD', 'MERCEDES', 'VW', 'OPEL', 'BMW', 'AUDI', 'TOYOTA', 'SKODA', 'HYUNDAI']

In [10]:
def _choose_brand_from_counts(counts: pd.Series, invalid_brand: str | None) -> str | None:
    """
    Given brand frequency counts for a group (e.g., a specific model or year)
    and the original invalid brand string, select a single brand according to:

    1. Take the highest frequency (the mode).
    2. If several brands tie for the highest frequency:
       - If exactly one of those brands contains the invalid brand as a
         substring, choose that one.
       - Otherwise, choose the first brand among the tied ones (i.e., the
         first in the value_counts order).

    If counts is empty, returns None.
    """
    if counts is None or len(counts) == 0:
        return None

    # value_counts is already sorted by frequency (desc) by default
    max_count = counts.iloc[0]
    top_brands = counts[counts == max_count].index.tolist()

    if len(top_brands) == 1:
        # Unique mode
        return top_brands[0]

    # There is a tie: try to break it using substring of the invalid brand
    if invalid_brand is None:
        invalid_upper = ""
    else:
        invalid_upper = str(invalid_brand).upper()

    substring_matches = [b for b in top_brands if invalid_upper and invalid_upper in str(b).upper()]

    if len(substring_matches) == 1:
        # Exactly one candidate contains the invalid brand string
        return substring_matches[0]

    # Either no substring match, or more than one → fall back to the first top brand
    return top_brands[0]


In [11]:
def fit_ambiguous_brand_resolver(
    train_df: pd.DataFrame,
    valid_brands: list[str],
    brand_col: str = "Brand",
    model_col: str = "model",
    year_col: str = "year",
) -> Dict[str, Any]:
    """
    Fit step for resolving ambiguous/invalid brands.

    This function learns, from the training data only:

    - The global most frequent valid brand (overall mode).
    - For each model, the frequency distribution of valid brands.
    - For each year, the frequency distribution of valid brands.

    These statistics are later used by the transform step to decide, for each
    invalid/ambiguous brand, which valid brand to choose.

    Assumptions:
    - `brand_col` has already been normalized (uppercased, accents removed, etc.).
    - `valid_brands` contains the canonical brand names that should be kept.
    """
    tmp = train_df.copy()

    valid_set = set(valid_brands)

    # Filter to rows where brand is valid
    valid_mask = tmp[brand_col].isin(valid_set)

    # Global most frequent valid brand
    brand_freq = tmp.loc[valid_mask, brand_col].value_counts()
    global_most_common_brand = brand_freq.index[0] if not brand_freq.empty else None

    # Brand frequency per year (only using valid brands)
    brand_counts_by_year: Dict[Any, pd.Series] = {}
    if year_col in tmp.columns:
        grouped_year = tmp.loc[valid_mask].groupby(year_col)[brand_col]
        for year_val, series in grouped_year:
            # series is the brand column for that year
            counts = series.value_counts()
            brand_counts_by_year[year_val] = counts

    # Brand frequency per model (only using valid brands)
    brand_counts_by_model: Dict[Any, pd.Series] = {}
    if model_col in tmp.columns:
        grouped_model = tmp.loc[valid_mask].groupby(model_col)[brand_col]
        for model_val, series in grouped_model:
            counts = series.value_counts()
            brand_counts_by_model[model_val] = counts

    state = {
        "brand_col": brand_col,
        "model_col": model_col,
        "year_col": year_col,
        "valid_brands": valid_set,
        "global_most_common_brand": global_most_common_brand,
        "brand_counts_by_year": brand_counts_by_year,
        "brand_counts_by_model": brand_counts_by_model,
    }

    return state


In [12]:
def transform_ambiguous_brands(
    df: pd.DataFrame,
    state: Dict[str, Any],
) -> tuple[pd.DataFrame, dict, list]:
    """
    Transform step for resolving ambiguous/invalid brands using
    statistics learned in `fit_ambiguous_brand_resolver`.

    For each row where the brand is not in `valid_brands`:

    - If `model` is known (and not 'UNKNOWN'):
        1. Look at the frequency distribution of valid brands for that model
           (from the training data).
        2. Choose the brand according to `_choose_brand_from_counts`.
        3. If no information is available for that model:
           - If original brand is 'UNKNOWN' -> use global most frequent valid brand.
           - Otherwise -> use the last replacement used for that invalid brand
             (past_replacements) or the global most frequent brand if none.

    - If `model` is missing or 'UNKNOWN':
        1. If both `year` is missing/empty AND original brand is 'UNKNOWN':
           -> directly use the global most frequent valid brand.
        2. Else, if `year` is present and we have statistics for that year:
           -> choose brand using `_choose_brand_from_counts`.
        3. If that still fails, use the global most frequent valid brand.

    Returns
    -------
    df_out : pd.DataFrame
        A copy of `df` with the brand column corrected.
    corrections : dict
        Mapping from (original_brand, model, year) to the chosen corrected brand.
    still_invalid : list
        List of brands that remain not in `valid_brands` and not equal to 'UNKNOWN'.
    """
    brand_col = state["brand_col"]
    model_col = state["model_col"]
    year_col = state["year_col"]
    valid_brands = state["valid_brands"]
    global_most_common_brand = state["global_most_common_brand"]
    brand_counts_by_year = state["brand_counts_by_year"]
    brand_counts_by_model = state["brand_counts_by_model"]

    df_out = df.copy()

    # Ensure no missing brands become a problem: treat NaN as 'UNKNOWN'
    df_out[brand_col] = df_out[brand_col].fillna("UNKNOWN")

    corrections: dict = {}
    past_replacements: dict = {}

    for idx, row in df_out.iterrows():
        original_brand = row[brand_col]
        brand_upper = str(original_brand).upper() if pd.notna(original_brand) else "UNKNOWN"

        # Skip already valid brands
        if brand_upper in valid_brands:
            continue

        model = row.get(model_col, None)
        year = row.get(year_col, None)

        corrected = None

        # ---------- Case 1: model is unknown or missing ----------
        if pd.isna(model) or str(model).strip().upper() == "UNKNOWN":

            # Special case: brand == 'UNKNOWN' and year is missing/empty
            if (pd.isna(year) or str(year).strip() == "") and brand_upper == "UNKNOWN":
                corrected = global_most_common_brand

            else:
                # Try using year-based statistics first
                if pd.notna(year) and year in brand_counts_by_year:
                    counts = brand_counts_by_year[year]
                    corrected = _choose_brand_from_counts(counts, brand_upper)

                # If still nothing, fall back to global most common brand
                if corrected is None:
                    corrected = global_most_common_brand

        # ---------- Case 2: model is known ----------
        else:
            if model in brand_counts_by_model:
                counts = brand_counts_by_model[model]
                corrected = _choose_brand_from_counts(counts, brand_upper)

            # No statistics available for this model
            if corrected is None:
                if brand_upper == "UNKNOWN":
                    # For UNKNOWN, go straight to global fallback
                    corrected = global_most_common_brand
                else:
                    # Try to be consistent with previous replacements for this invalid
                    corrected = past_replacements.get(brand_upper, global_most_common_brand)

        # Apply the correction
        df_out.at[idx, brand_col] = corrected
        corrections[(original_brand, model, year)] = corrected
        past_replacements[brand_upper] = corrected

        # Optional: logging (comment out if not needed)
        # if corrected == "UNKNOWN":
        #     print(f"FINAL RESULT: '{original_brand}' (model='{model}', year='{year}') - remained 'UNKNOWN'")
        # else:
        #     print(f"FINAL RESULT: '{original_brand}' (model='{model}', year='{year}') - corrected to '{corrected}'")

    # Brands that are still not valid (and not 'UNKNOWN') after correction
    still_invalid = [
        b for b in df_out[brand_col].unique()
        if b not in valid_brands and b != "UNKNOWN"
    ]

    return df_out, corrections, still_invalid


### Model


# 2 Numerical features

In [13]:
# basic function before cross val 

def basic_num_prep(
        series: pd.Series,
        clip_max: float | None = None,
        min_value: float | None = None,
        do_abs: bool = True,
        flooring: bool = False,
        rounding: bool = False,
        round_decimals: int = 0,
) -> pd.Series:
    
    """
    """ 
    
    # ensures we are working with numeric data
    s = pd.to_numeric(series, errors="coerce").copy()

    # absolute value transformation 
    if do_abs:
        s = s.abs()
    
    # cliiping maximum
    if clip_max is not None:
        s = s.clip(upper=clip_max)
    
    # enforce minimum
    if min_value is not None:
        s = s.clip(lower=min_val)
    
    if flooring:
        s = np.floor(s)
    elif rounding:
        s = s.round(round_decimals)
    
    return s

In [14]:
# dropping carid and has damage
#....

# inside each fold


### year

In [15]:
# Year 
# fit que calcula a mediana agrupada por model e caso seja nula
# usa a mediana global

#transform: aplica a median; as outras transformações serão feitas antes no coiso

def fit_year_median(
        train_df: pd.DataFrame,
        year_col: str = "year",
        model_col: str = "model",
        max_valid_year: int = 2020,
):
    tmp = train_df.copy()

    # ensure numeric 
    tmp[year_col] = pd.to_numeric(tmp[year_col], errors="coerce")

    # cap to maximum allowed year, already performed before but for safety 
    tmp.loc[tmp[year_col] > max_valid_year, year_col] = max_valid_year

    # floor to int already done before too
    tmp[year_col] = np.floor(tmp[year_col])

    # computed medians
    global_year_median = tmp[year_col].median()
    model_year_median = tmp.groupby(model_col)[year_col].median()

    state = {
        "year_col": year_col,
        "model_col": model_col,
        "max_valid_year":  max_valid_year,
        "global_year_median": global_year_median,
        "model_year_median": model_year_median
    }

    return state


In [16]:
def transform_year_with_model_median(
        df: pd.DataFrame,
        state: dict,
) -> pd.DataFrame:
    """
    """
    year_col = state["year_col"]
    model_col = state["model_col"]
    max_valid_year = state["max_valid_year"]
    global_median = state["global_year_median"]
    model_median = state["model_year_median"]

    df_out = df.copy()

    # 1) Ensure numeric
    df_out[year_col] = pd.to_numeric(df_out[year_col], errors="coerce")

    # 2) Cap to max_valid_year
    df_out.loc[df_out[year_col] > max_valid_year, year_col] = max_valid_year

    # 3) Floor to integer
    df_out[year_col] = np.floor(df_out[year_col])

    # 4) Impute missing years:
    #    - first use model-specific median
    df_out[year_col] = df_out[year_col].fillna(
        df_out[model_col].map(model_median)
    )
    #    - then use global median as fallback
    df_out[year_col] = df_out[year_col].fillna(global_median)

    return df_out

### previousOwners

In [17]:
def fit_previous_owners_imputer(
    train_df: pd.DataFrame,
    owners_col: str = "previousOwners",
    year_col: str = "year",
    mileage_col: str = "mileage",
    mileage_threshold: float = 15000.0,
    min_years_since_registration: float = 2.0,
):
    """
    Fit step for previousOwners preprocessing.

    Assumes that `year` and `mileage` have already been preprocessed.

    Steps on train_df:
    1. Convert previousOwners to numeric (invalid -> NaN).
    2. Take absolute value (negatives become positive).
    3. Round to nearest integer.
    4. Apply "zero imprecision correction":
       If ((max_year - year) > min_years_since_registration OR mileage > mileage_threshold)
       AND previousOwners == 0 -> set previousOwners = 1.
    5. Compute the median of previousOwners (after the above cleaning, before imputation).

    Returns
    -------
    state : dict
        Dictionary containing all parameters needed for the transform step:
        - 'owners_col': name of previous owners column
        - 'year_col': name of year column
        - 'mileage_col': name of mileage column
        - 'max_year': max year observed in train
        - 'mileage_threshold': threshold used for mileage condition
        - 'min_years_since_registration': threshold for age condition
        - 'imputation_median': median of cleaned previousOwners (train only)
    """
    tmp = train_df.copy()

    # 1) Convert to numeric
    tmp[owners_col] = pd.to_numeric(tmp[owners_col], errors="coerce")

    # 2) Absolute values
    tmp[owners_col] = tmp[owners_col].abs()

    # 3) Round to nearest integer
    tmp[owners_col] = tmp[owners_col].round()

    # Compute max_year based on train only
    max_year = tmp[year_col].max()

    # 4) Zero imprecision correction
    #    If car is "old enough" or has enough mileage, a 0-owners entry is suspicious.
    #    Condition: ((max_year - year) > min_years_since_registration) OR (mileage > mileage_threshold)
    #               AND previousOwners == 0
    inac_own_condition_train = (
        ((max_year - tmp[year_col]) > min_years_since_registration)
        | (tmp[mileage_col] > mileage_threshold)
    ) & (tmp[owners_col] == 0)

    tmp.loc[inac_own_condition_train, owners_col] = 1

    # 5) Compute median after cleaning (NaNs are skipped)
    imputation_median = tmp[owners_col].median(skipna=True)


    state = {
        "owners_col": owners_col,
        "year_col": year_col,
        "mileage_col": mileage_col,
        "max_year": max_year,
        "mileage_threshold": mileage_threshold,
        "min_years_since_registration": min_years_since_registration,
        "imputation_median": imputation_median,
    }

    return state



def transform_previous_owners_imputer(
    df: pd.DataFrame,
    state: dict,
) -> pd.DataFrame:
    """
    Transform step for previousOwners preprocessing using fitted statistics.

    Applies on any DataFrame (train, validation, test):
    1. Convert previousOwners to numeric (invalid -> NaN).
    2. Take absolute value (negatives become positive).
    3. Round to nearest integer.
    4. Apply "zero imprecision correction" using the `max_year` learned in fit:
       If ((max_year - year) > min_years_since_registration OR mileage > mileage_threshold)
       AND previousOwners == 0 -> set previousOwners = 1.
    5. Impute remaining NaNs with the median learned from train.

    Parameters
    ----------
    df : pd.DataFrame
        Any DataFrame to be transformed.
    state : dict
        Dictionary returned by `fit_previous_owners_imputer`.

    Returns
    -------
    pd.DataFrame
        Copy of df with the previousOwners column cleaned and imputed.
    """
    owners_col = state["owners_col"]
    year_col = state["year_col"]
    mileage_col = state["mileage_col"]
    max_year = state["max_year"]
    mileage_threshold = state["mileage_threshold"]
    min_years_since_registration = state["min_years_since_registration"]
    imputation_median = state["imputation_median"]

    df_out = df.copy()

    # 1) Convert to numeric
    df_out[owners_col] = pd.to_numeric(df_out[owners_col], errors="coerce")

    # 2) Absolute values
    df_out[owners_col] = df_out[owners_col].abs()

    # 3) Round to nearest integer
    df_out[owners_col] = df_out[owners_col].round()

    # 4) Zero imprecision correction (using max_year from TRAIN ONLY)
    inac_own_condition = (
        ((max_year - df_out[year_col]) > min_years_since_registration)
        | (df_out[mileage_col] > mileage_threshold)
    ) & (df_out[owners_col] == 0)

    df_out.loc[inac_own_condition, owners_col] = 1

    # 5) Impute NaNs with train-based median
    df_out[owners_col] = df_out[owners_col].fillna(imputation_median)

    return df_out

### Paint quality

In [18]:
def fit_paint_quality_imputer(
    train_df: pd.DataFrame,
    paint_col: str = "paintQuality%",
    clip_max: float = 100.0,
    do_abs: bool = True,
):
    """
    Fit step for paintQuality% preprocessing.

    Steps applied on train_df:
    1. Convert paint quality column to numeric (invalid values -> NaN).
    2. Optionally take absolute value (negatives become positive).
    3. Clip values above `clip_max` to `clip_max`.
    4. Compute the median of the cleaned paint quality column (train only).

    Parameters
    ----------
    train_df : pd.DataFrame
        Training DataFrame.
    paint_col : str, default "paintQuality%%"
        Name of the paint quality column.
    clip_max : float, default 100.0
        Maximum allowed value for paint quality.
    do_abs : bool, default True
        If True, apply absolute value to the column.

    Returns
    -------
    state : dict
        Dictionary containing:
        - 'paint_col': name of paint quality column
        - 'clip_max': clipping value used
        - 'do_abs': whether abs() was applied
        - 'imputation_median': median of cleaned paint quality (train only)
    """
    tmp = train_df.copy()

    # 1) Convert to numeric
    tmp[paint_col] = pd.to_numeric(tmp[paint_col], errors="coerce")

    # 2) Absolute values (optionally)
    if do_abs:
        tmp[paint_col] = tmp[paint_col].abs()

    # 3) Clip upper bound
    tmp[paint_col] = tmp[paint_col].clip(upper=clip_max)

    # 4) Median after cleaning (NaNs skipped)
    imputation_median = tmp[paint_col].median(skipna=True)

    state = {
        "paint_col": paint_col,
        "clip_max": clip_max,
        "do_abs": do_abs,
        "imputation_median": imputation_median,
    }

    return state

In [19]:
def transform_paint_quality_imputer(
    df: pd.DataFrame,
    state: dict,
) -> pd.DataFrame:
    """
    Transform step for paintQuality% preprocessing using fitted statistics.

    Applies on any DataFrame (train, validation, test):
    1. Convert paint quality column to numeric (invalid values -> NaN).
    2. Optionally take absolute value (depending on `state['do_abs']`).
    3. Clip values above `clip_max` (from fit).
    4. Impute remaining NaNs with the median learned from train.

    Parameters
    ----------
    df : pd.DataFrame
        Any DataFrame to be transformed.
    state : dict
        Dictionary returned by `fit_paint_quality_imputer`.

    Returns
    -------
    pd.DataFrame
        Copy of df with the paint quality column cleaned and imputed.
    """
    paint_col = state["paint_col"]
    clip_max = state["clip_max"]
    do_abs = state["do_abs"]
    imputation_median = state["imputation_median"]

    df_out = df.copy()

    # 1) Convert to numeric
    df_out[paint_col] = pd.to_numeric(df_out[paint_col], errors="coerce")

    # 2) Absolute values (if used in fit)
    if do_abs:
        df_out[paint_col] = df_out[paint_col].abs()

    # 3) Clip upper bound
    df_out[paint_col] = df_out[paint_col].clip(upper=clip_max)

    # 4) Impute NaNs with train-based median
    df_out[paint_col] = df_out[paint_col].fillna(imputation_median)

    return df_out


### mileage

In [20]:
def fit_mileage_imputer(
    train_df: pd.DataFrame,
    mileage_col: str = "mileage",
    do_abs: bool = True,
):
    """
    Fit step for mileage preprocessing.

    Steps applied on train_df:
    1. Convert mileage column to numeric (invalid values -> NaN).
    2. Optionally convert negative values to absolute.
    3. Compute the median of the cleaned mileage column (train only).

    Parameters
    ----------
    train_df : pd.DataFrame
        Training DataFrame.
    mileage_col : str, default "mileage"
        Name of the mileage column.
    do_abs : bool, default True
        If True, apply absolute value to the mileage column.

    Returns
    -------
    state : dict
        Dictionary containing:
        - 'mileage_col': name of mileage column
        - 'do_abs': whether abs() was applied
        - 'imputation_median': median of cleaned mileage (train only)
    """
    tmp = train_df.copy()

    # 1) Convert to numeric (invalid values -> NaN)
    tmp[mileage_col] = pd.to_numeric(tmp[mileage_col], errors="coerce")

    # 2) Absolute values (if requested)
    if do_abs:
        tmp[mileage_col] = tmp[mileage_col].abs()

    # 3) Median after cleaning (NaNs skipped)
    imputation_median = tmp[mileage_col].median(skipna=True)

    state = {
        "mileage_col": mileage_col,
        "do_abs": do_abs,
        "imputation_median": imputation_median,
    }

    return state


In [21]:
def transform_mileage_imputer(
    df: pd.DataFrame,
    state: dict,
) -> pd.DataFrame:
    """
    Transform step for mileage preprocessing using fitted statistics.

    Applies on any DataFrame (train, validation, test):
    1. Convert mileage column to numeric (invalid values -> NaN).
    2. Optionally convert negative values to absolute (same as in fit).
    3. Impute remaining NaNs with the median learned from train.

    Parameters
    ----------
    df : pd.DataFrame
        Any DataFrame to be transformed.
    state : dict
        Dictionary returned by `fit_mileage_imputer`.

    Returns
    -------
    pd.DataFrame
        Copy of df with the mileage column cleaned and imputed.
    """
    mileage_col = state["mileage_col"]
    do_abs = state["do_abs"]
    imputation_median = state["imputation_median"]

    df_out = df.copy()

    # 1) Convert to numeric
    df_out[mileage_col] = pd.to_numeric(df_out[mileage_col], errors="coerce")

    # 2) Absolute values (if used in fit)
    if do_abs:
        df_out[mileage_col] = df_out[mileage_col].abs()

    # 3) Impute NaNs with train-based median
    df_out[mileage_col] = df_out[mileage_col].fillna(imputation_median)

    return df_out

### tax THIS ONE NEEDS FURTHER IMPROVEMENT

In [22]:
def fit_tax_imputer(
    train_df: pd.DataFrame,
    tax_col: str = "tax",
    do_abs: bool = True,
):
    """
    Fit step for tax preprocessing.

    Steps applied on train_df:
    1. Convert tax column to numeric (invalid values -> NaN).
    2. Optionally convert negative values to absolute.
    3. Compute the median of the cleaned tax column (train only).

    Parameters
    ----------
    train_df : pd.DataFrame
        Training DataFrame.
    tax_col : str, default "tax"
        Name of the tax column.
    do_abs : bool, default True
        If True, apply absolute value to the tax column.

    Returns
    -------
    state : dict
        Dictionary containing:
        - 'tax_col': name of tax column
        - 'do_abs': whether abs() was applied
        - 'imputation_median': median of cleaned tax (train only)
    """
    tmp = train_df.copy()

    # 1) Convert to numeric (invalid values -> NaN)
    tmp[tax_col] = pd.to_numeric(tmp[tax_col], errors="coerce")

    # 2) Absolute values (if requested)
    if do_abs:
        tmp[tax_col] = tmp[tax_col].abs()

    # 3) Median after cleaning (NaNs skipped)
    imputation_median = tmp[tax_col].median(skipna=True)

    state = {
        "tax_col": tax_col,
        "do_abs": do_abs,
        "imputation_median": imputation_median,
    }

    return state


In [23]:
def transform_tax_imputer(
    df: pd.DataFrame,
    state: dict,
) -> pd.DataFrame:
    """
    Transform step for tax preprocessing using fitted statistics.

    Applies on any DataFrame (train, validation, test):
    1. Convert tax column to numeric (invalid values -> NaN).
    2. Optionally convert negative values to absolute (same as in fit).
    3. Impute remaining NaNs with the median learned from train.

    Parameters
    ----------
    df : pd.DataFrame
        Any DataFrame to be transformed.
    state : dict
        Dictionary returned by `fit_tax_imputer`.

    Returns
    -------
    pd.DataFrame
        Copy of df with the tax column cleaned and imputed.
    """
    tax_col = state["tax_col"]
    do_abs = state["do_abs"]
    imputation_median = state["imputation_median"]

    df_out = df.copy()

    # 1) Convert to numeric
    df_out[tax_col] = pd.to_numeric(df_out[tax_col], errors="coerce")

    # 2) Absolute values (if used in fit)
    if do_abs:
        df_out[tax_col] = df_out[tax_col].abs()

    # 3) Impute NaNs with train-based median
    df_out[tax_col] = df_out[tax_col].fillna(imputation_median)

    return df_out

### mpg

I think this one could be improved too

In [24]:
def fit_mpg_imputer(
    train_df: pd.DataFrame,
    mpg_col: str = "mpg",
    do_abs: bool = True,
    clip_lower: float = 10.0,
    clip_upper: float = 200.0,
):
    """
    Fit step for mpg preprocessing.

    Steps applied on train_df:
    1. Convert mpg column to numeric (invalid values -> NaN).
    2. Optionally convert negative values to absolute.
    3. Compute the median of the cleaned mpg column (train only),
       BEFORE any clipping.

    Parameters
    ----------
    train_df : pd.DataFrame
        Training DataFrame.
    mpg_col : str, default "mpg"
        Name of the mpg column.
    do_abs : bool, default True
        If True, apply absolute value to the mpg column.
    clip_lower : float, default 10.0
        Lower bound used later for clipping (stored for consistency).
    clip_upper : float, default 200.0
        Upper bound used later for clipping (stored for consistency).

    Returns
    -------
    state : dict
        Dictionary containing:
        - 'mpg_col': name of mpg column
        - 'do_abs': whether abs() was applied
        - 'clip_lower': lower bound for clipping
        - 'clip_upper': upper bound for clipping
        - 'imputation_median': median of cleaned mpg (train only, pre-clip)
    """
    tmp = train_df.copy()

    # 1) Convert to numeric
    tmp[mpg_col] = pd.to_numeric(tmp[mpg_col], errors="coerce")

    # 2) Absolute values (if requested)
    if do_abs:
        tmp[mpg_col] = tmp[mpg_col].abs()

    # 3) Median before clipping
    imputation_median = tmp[mpg_col].median(skipna=True)

    state = {
        "mpg_col": mpg_col,
        "do_abs": do_abs,
        "clip_lower": clip_lower,
        "clip_upper": clip_upper,
        "imputation_median": imputation_median,
    }

    return state

In [25]:
def transform_mpg_imputer(
    df: pd.DataFrame,
    state: dict,
) -> pd.DataFrame:
    """
    Transform step for mpg preprocessing using fitted statistics.

    Applies on any DataFrame (train, validation, test):

    1. Convert mpg column to numeric (invalid values -> NaN).
    2. Optionally convert negative values to absolute (same as in fit).
    3. Impute NaNs with the median learned from train.
    4. Clip mpg values between clip_lower and clip_upper (inclusive).

    Parameters
    ----------
    df : pd.DataFrame
        Any DataFrame to be transformed.
    state : dict
        Dictionary returned by `fit_mpg_imputer`.

    Returns
    -------
    pd.DataFrame
        Copy of df with the mpg column cleaned, imputed and clipped.
    """
    mpg_col = state["mpg_col"]
    do_abs = state["do_abs"]
    clip_lower = state["clip_lower"]
    clip_upper = state["clip_upper"]
    imputation_median = state["imputation_median"]

    df_out = df.copy()

    # 1) Convert to numeric
    df_out[mpg_col] = pd.to_numeric(df_out[mpg_col], errors="coerce")

    # 2) Absolute values (if used in fit)
    if do_abs:
        df_out[mpg_col] = df_out[mpg_col].abs()

    # 3) Impute NaNs with train-based median
    df_out[mpg_col] = df_out[mpg_col].fillna(imputation_median)

    # 4) Clip to the specified range
    df_out[mpg_col] = df_out[mpg_col].clip(lower=clip_lower, upper=clip_upper)

    return df_out


### engine size

In [26]:
def fit_engine_size_imputer(
    train_df: pd.DataFrame,
    engine_col: str = "engineSize",
    do_abs: bool = True,
    treat_zero_as_nan: bool = True,
):
    """
    Fit step for engineSize preprocessing.

    Steps applied on train_df:
    1. Convert engine size column to numeric (invalid values -> NaN).
    2. Optionally convert negative values to absolute.
    3. Optionally treat 0 values as NaN (so they are imputed).
    4. Compute the median of the cleaned engine size column (train only).

    Parameters
    ----------
    train_df : pd.DataFrame
        Training DataFrame.
    engine_col : str, default "engineSize"
        Name of the engine size column.
    do_abs : bool, default True
        If True, apply absolute value to the engine size column.
    treat_zero_as_nan : bool, default True
        If True, replace exact zeros with NaN before computing the median.

    Returns
    -------
    state : dict
        Dictionary containing:
        - 'engine_col': name of engine size column
        - 'do_abs': whether abs() was applied
        - 'treat_zero_as_nan': whether 0 was treated as NaN
        - 'imputation_median': median of cleaned engine size (train only)
    """
    tmp = train_df.copy()

    # 1) Convert to numeric
    tmp[engine_col] = pd.to_numeric(tmp[engine_col], errors="coerce")

    # 2) Absolute values (if requested)
    if do_abs:
        tmp[engine_col] = tmp[engine_col].abs()

    # 3) Treat zeros as NaN (if requested)
    if treat_zero_as_nan:
        tmp.loc[tmp[engine_col] == 0, engine_col] = np.nan

    # 4) Median after cleaning (NaNs skipped)
    imputation_median = tmp[engine_col].median(skipna=True)

    state = {
        "engine_col": engine_col,
        "do_abs": do_abs,
        "treat_zero_as_nan": treat_zero_as_nan,
        "imputation_median": imputation_median,
    }

    return state

In [27]:
def transform_engine_size_imputer(
    df: pd.DataFrame,
    state: dict,
) -> pd.DataFrame:
    """
    Transform step for engineSize preprocessing using fitted statistics.

    Applies on any DataFrame (train, validation, test):

    1. Convert engine size column to numeric (invalid values -> NaN).
    2. Optionally convert negative values to absolute (same as in fit).
    3. Optionally treat 0 values as NaN (same rule as in fit).
    4. Impute remaining NaNs with the median learned from train.

    Parameters
    ----------
    df : pd.DataFrame
        Any DataFrame to be transformed.
    state : dict
        Dictionary returned by `fit_engine_size_imputer`.

    Returns
    -------
    pd.DataFrame
        Copy of df with the engine size column cleaned and imputed.
    """
    engine_col = state["engine_col"]
    do_abs = state["do_abs"]
    treat_zero_as_nan = state["treat_zero_as_nan"]
    imputation_median = state["imputation_median"]

    df_out = df.copy()

    # 1) Convert to numeric
    df_out[engine_col] = pd.to_numeric(df_out[engine_col], errors="coerce")

    # 2) Absolute values (if used in fit)
    if do_abs:
        df_out[engine_col] = df_out[engine_col].abs()

    # 3) Treat zeros as NaN (if used in fit)
    if treat_zero_as_nan:
        df_out.loc[df_out[engine_col] == 0, engine_col] = np.nan

    # 4) Impute NaNs with train-based median
    df_out[engine_col] = df_out[engine_col].fillna(imputation_median)

    return df_out


## models

In [28]:
def _choose_model_from_counts(counts: pd.Series, invalid_model: str | None) -> str | None:
    """
    Given model frequency counts for a group (e.g., a specific context)
    and the original invalid model string, select a single model according to:

    1. Take the highest frequency (the mode).
    2. If several models tie for the highest frequency:
       - If exactly one of those models contains the invalid model as a
         substring, choose that one.
       - Otherwise, choose the first model among the tied ones
         (i.e., the first in value_counts order).

    If counts is empty, returns None.
    """
    if counts is None or len(counts) == 0:
        return None

    max_count = counts.iloc[0]
    top_models = counts[counts == max_count].index.tolist()

    if len(top_models) == 1:
        return top_models[0]

    invalid_upper = (str(invalid_model).upper()
                     if invalid_model is not None else "")

    substring_matches = [
        m for m in top_models
        if invalid_upper and invalid_upper in str(m).upper()
    ]

    if len(substring_matches) == 1:
        return substring_matches[0]

    # Either no substring match, or more than one → fall back to the first top model
    return top_models[0]

In [29]:
def fit_invalid_model_resolver(
    train_df: pd.DataFrame,
    valid_models_by_brand: Dict[str, list],
    brand_col: str = "Brand",
    model_col: str = "model",
    year_col: str = "year",
    fuel_col: str = "fuelType",
    mpg_col: str = "mpg",
) -> Dict[str, Any]:
    """
    Fit step for resolving invalid/ambiguous models.

    Assumes that:
    - basic string normalization has already been applied to `model_col`
      and `brand_col` (upper, strip, accents removed).
    - NaN in `model_col` have already been converted to 'UNKNOWN' (or
      equivalent deterministic step).

    This function learns, from TRAIN ONLY:

    1. A consolidated dictionary of valid models per brand
       (starting from `valid_models_by_brand`, possibly extended).
    2. The global set of all valid models.
    3. A mapping invalid_model -> chosen_model (for invalid strings
       observed in the training set, excluding 'UNKNOWN').
    4. Mean mpg per model (for mpg-based tie-breaking).
    5. Context-based maps for inferring 'UNKNOWN' models:
       - (Brand, year, fuelType) -> mode(model)
       - (Brand, year) -> mode(model)
       - (Brand, fuelType) -> mode(model)
       - Brand -> mode(model)
       (all computed only from rows where model != 'UNKNOWN').
    """
    tmp = train_df.copy()

    # Ensure model is a clean string with 'UNKNOWN' where needed
    tmp[model_col] = (
        tmp[model_col]
        .fillna("UNKNOWN")
        .astype(str)
        .str.strip()
        .str.upper()
    )

    tmp[brand_col] = (
        tmp[brand_col]
        .astype(str)
        .str.strip()
        .str.upper()
    )

    # Deep copy of valid_models_by_brand to avoid mutating the original dict
    valid_models_by_brand_fit: Dict[str, list] = {
        b: list(models) for b, models in valid_models_by_brand.items()
    }
    brand_valid_sets: Dict[str, set] = {
        b: set(models) for b, models in valid_models_by_brand_fit.items()
    }

    # Global set of all valid models
    all_valid_models = set(
        m for models in valid_models_by_brand_fit.values() for m in models
    )

    # Mean mpg per model (only for rows with valid model)
    if mpg_col in tmp.columns:
        mpg_means = (
            tmp[tmp[model_col].isin(all_valid_models)]
            .groupby(model_col)[mpg_col]
            .mean()
        )
    else:
        mpg_means = pd.Series(dtype=float)

    # --------- Context maps for UNKNOWN model inference (TRAIN ONLY) ---------
    known_mask = tmp[model_col] != "UNKNOWN"
    known = tmp[known_mask]

    ctx_brand_year_fuel: Dict[tuple, str] = {}
    ctx_brand_year: Dict[tuple, str] = {}
    ctx_brand_fuel: Dict[tuple, str] = {}
    ctx_brand: Dict[str, str] = {}

    # (Brand, year, fuelType)
    if all(c in known.columns for c in [brand_col, year_col, fuel_col]):
        grp = known.groupby([brand_col, year_col, fuel_col])[model_col]
        for key, series in grp:
            mode = series.mode()
            if not mode.empty:
                ctx_brand_year_fuel[key] = mode.iloc[0]

    # (Brand, year)
    if all(c in known.columns for c in [brand_col, year_col]):
        grp = known.groupby([brand_col, year_col])[model_col]
        for key, series in grp:
            mode = series.mode()
            if not mode.empty:
                ctx_brand_year[key] = mode.iloc[0]

    # (Brand, fuelType)
    if all(c in known.columns for c in [brand_col, fuel_col]):
        grp = known.groupby([brand_col, fuel_col])[model_col]
        for key, series in grp:
            mode = series.mode()
            if not mode.empty:
                ctx_brand_fuel[key] = mode.iloc[0]

    # Brand only
    grp = known.groupby(brand_col)[model_col]
    for b, series in grp:
        mode = series.mode()
        if not mode.empty:
            ctx_brand[b] = mode.iloc[0]

    # --------- Mapping invalid_model -> chosen_model (TRAIN OBSERVED) ---------
    invalid_models = sorted(
        [
            m for m in tmp[model_col].unique()
            if m not in all_valid_models and m != "UNKNOWN"
        ],
        key=len,
        reverse=True,
    )

    invalid_to_model: Dict[str, str] = {}

    for invalid in invalid_models:
        subset = tmp[tmp[model_col] == invalid]
        if subset.empty:
            continue

        # Dominant brand for this invalid model
        brand_mode = subset[brand_col].mode()
        if brand_mode.empty:
            continue
        brand = brand_mode.iloc[0]

        brand_valids = brand_valid_sets.get(brand, set())

        # Check substring matches in known valid models for this brand
        matches = [vm for vm in brand_valids if invalid in vm]

        # Case 1: exactly one substring match -> treat as typo
        if len(matches) == 1:
            chosen = matches[0]

        # Case 2: no matches -> promote invalid as new valid model for this brand
        elif len(matches) == 0:
            chosen = invalid
            valid_models_by_brand_fit.setdefault(brand, []).append(invalid)
            brand_valid_sets.setdefault(brand, set()).add(invalid)
            all_valid_models.add(invalid)

        # Case 3: multiple matches -> use mpg heuristic on TRAIN
        else:
            chosen = None
            if mpg_col in subset.columns and subset[mpg_col].notna().any() and not mpg_means.empty:
                invalid_mean_mpg = subset[mpg_col].mean(skipna=True)
                candidate_means = {
                    m: mpg_means[m]
                    for m in matches
                    if m in mpg_means.index
                }
                if candidate_means:
                    chosen = min(
                        candidate_means.keys(),
                        key=lambda m: abs(candidate_means[m] - invalid_mean_mpg)
                    )

            if chosen is None:
                # Fallback: first candidate
                chosen = matches[0]

        invalid_to_model[invalid] = chosen

    state = {
        "brand_col": brand_col,
        "model_col": model_col,
        "year_col": year_col,
        "fuel_col": fuel_col,
        "mpg_col": mpg_col,
        "valid_models_by_brand": valid_models_by_brand_fit,
        "all_valid_models": all_valid_models,
        "invalid_to_model": invalid_to_model,
        "mpg_means": mpg_means,
        "ctx_brand_year_fuel": ctx_brand_year_fuel,
        "ctx_brand_year": ctx_brand_year,
        "ctx_brand_fuel": ctx_brand_fuel,
        "ctx_brand": ctx_brand,
    }

    return state


In [30]:
def transform_invalid_models(
    df: pd.DataFrame,
    state: Dict[str, Any],
) -> tuple[pd.DataFrame, dict, list]:
    """
    Transform step for invalid/ambiguous model resolution, using the
    statistics learned in `fit_invalid_model_resolver`.

    Workflow for each row:

    1. If model is already in the global set of valid models -> keep as is.
    2. Else if model == 'UNKNOWN':
       - Try to infer a model using context maps from TRAIN, in order:
         (Brand, year, fuelType) -> (Brand, year) -> (Brand, fuelType) -> Brand.
       - If nothing is found, keep 'UNKNOWN'.
    3. Else (non-'UNKNOWN' invalid model string):
       - If this invalid model was seen in training:
         * Replace by `invalid_to_model[invalid]`.
       - If unseen in training:
         * Try substring matches against valid models of that brand
           (from `valid_models_by_brand` in state).
           - If exactly one match -> use it.
           - If multiple matches:
             + If mpg is available and model means are known -> pick the
               candidate whose mean mpg is closest to the row's mpg.
             + Otherwise, pick the first candidate.
           - If no matches -> leave as is.

    Returns
    -------
    df_out : pd.DataFrame
        Copy of df with corrected models.
    corrections : dict
        Mapping from (original_model, brand, year, fuel, row_index) to
        the chosen corrected model.
    still_invalid : list
        List of model strings that remain outside the global valid set
        and are not 'UNKNOWN'.
    """
    brand_col = state["brand_col"]
    model_col = state["model_col"]
    year_col = state["year_col"]
    fuel_col = state["fuel_col"]
    mpg_col = state["mpg_col"]

    valid_models_by_brand = state["valid_models_by_brand"]
    all_valid_models = state["all_valid_models"]
    invalid_to_model = state["invalid_to_model"]
    mpg_means = state["mpg_means"]

    ctx_brand_year_fuel = state["ctx_brand_year_fuel"]
    ctx_brand_year = state["ctx_brand_year"]
    ctx_brand_fuel = state["ctx_brand_fuel"]
    ctx_brand = state["ctx_brand"]

    df_out = df.copy()

    # Normalize brand/model here as a safety net
    df_out[model_col] = (
        df_out[model_col]
        .fillna("UNKNOWN")
        .astype(str)
        .str.strip()
        .str.upper()
    )
    df_out[brand_col] = (
        df_out[brand_col]
        .astype(str)
        .str.strip()
        .str.upper()
    )

    corrections: dict = {}

    for idx, row in df_out.iterrows():
        original_model = row[model_col]
        brand = row.get(brand_col, None)
        year = row.get(year_col, None)
        fuel = row.get(fuel_col, None)
        mpg_value = row.get(mpg_col, None)

        model_upper = str(original_model).upper() if pd.notna(original_model) else "UNKNOWN"

        # Case 0: already valid -> skip
        if model_upper in all_valid_models:
            continue

        corrected = model_upper  # default: keep as is

        # ---------- Case 1: 'UNKNOWN' model ----------
        if model_upper == "UNKNOWN":
            key3 = (brand, year, fuel)
            key2_by = (brand, year)
            key2_bf = (brand, fuel)

            if key3 in ctx_brand_year_fuel:
                corrected = ctx_brand_year_fuel[key3]
            elif key2_by in ctx_brand_year:
                corrected = ctx_brand_year[key2_by]
            elif key2_bf in ctx_brand_fuel:
                corrected = ctx_brand_fuel[key2_bf]
            elif brand in ctx_brand:
                corrected = ctx_brand[brand]
            else:
                corrected = "UNKNOWN"  # keep unknown if no context helps

        # ---------- Case 2: non-'UNKNOWN' invalid model ----------
        else:
            # Seen in training -> direct mapping
            if model_upper in invalid_to_model:
                corrected = invalid_to_model[model_upper]
            else:
                # Unseen invalid: try substring matches within brand-valid models
                brand_valids = set(valid_models_by_brand.get(brand, []))
                matches = [vm for vm in brand_valids if model_upper in vm]

                if len(matches) == 1:
                    corrected = matches[0]
                elif len(matches) > 1:
                    chosen = None
                    if pd.notna(mpg_value) and not mpg_means.empty:
                        candidate_means = {
                            m: mpg_means[m]
                            for m in matches
                            if m in mpg_means.index
                        }
                        if candidate_means:
                            chosen = min(
                                candidate_means.keys(),
                                key=lambda m: abs(candidate_means[m] - mpg_value)
                            )
                    if chosen is None:
                        chosen = matches[0]
                    corrected = chosen
                else:
                    # No match in known models for this brand -> keep as is
                    corrected = model_upper

        # Apply correction if it changed
        if corrected != model_upper:
            df_out.at[idx, model_col] = corrected

        corrections[(original_model, brand, year, fuel, idx)] = corrected

    # Models still not in the valid set and not UNKNOWN
    still_invalid = [
        m for m in df_out[model_col].unique()
        if m not in all_valid_models and m != "UNKNOWN"
    ]

    return df_out, corrections, still_invalid


### fuel type 

In [31]:
def fit_fueltype_resolver(
    train_df: pd.DataFrame,
    valid_fueltypes: list[str],
    fuel_col: str = "fuelType",
    brand_col: str = "Brand",
    model_col: str = "model",
    transm_col: str = "transmission",
) -> Dict[str, Any]:
    """
    Fit step for resolving invalid/ambiguous fuel types.

    Learns from TRAIN ONLY:
    -----------------------------------------
    1. A set of valid fuel types (starting from the given list,
       possibly expanded with new valid strings observed in train).
    2. For each invalid fuel (non-UNKNOWN):
       - substring match corrections,
       - or promotion to new valid fuel type.
    3. Context statistics for resolving UNKNOWN:
       a) (Brand, Model, Transmission) -> mode(fuel)
       b) (Brand, Model) -> mode(fuel)
       c) Brand -> mode(fuel)

    Assumes the fuel column is already normalized (UPPERCASE, stripped, NaN→UNKNOWN).
    """
    tmp = train_df.copy()

    # Normalize the base valid list
    valid_fueltypes = [f.upper().strip() for f in valid_fueltypes]
    valid_set = set(valid_fueltypes)

    # Identify invalid fuel types (seen in train)
    invalids = sorted(
        [f for f in tmp[fuel_col].unique() if f not in valid_set],
        key=len,
        reverse=True
    )

    # Mapping invalid_fuel -> corrected_fuel
    invalid_to_valid: Dict[str, str] = {}

    # Expandable set (we may promote new valid fuel types)
    dynamic_valids = set(valid_set)

    # Learn corrections for each invalid fuel type observed in training
    for invalid in invalids:
        if invalid == "UNKNOWN":
            continue

        subset = tmp[tmp[fuel_col] == invalid]
        if subset.empty:
            continue

        # Substring matching with known valid fueltypes
        matches = [v for v in valid_fueltypes if invalid in v]

        if len(matches) == 1:
            # Unique match -> typo fix
            chosen = matches[0]

        elif len(matches) == 0:
            # No match -> promote new valid entry
            chosen = invalid
            dynamic_valids.add(invalid)
            valid_fueltypes.append(invalid)

        else:
            # Multiple matches -> pick the most frequent among them
            # (based on TRAIN only rows that contain any of these matches)
            df_f = tmp[tmp[fuel_col].isin(matches)]
            if not df_f.empty:
                chosen = df_f[fuel_col].mode().iloc[0]
            else:
                chosen = matches[0]   # fallback

        invalid_to_valid[invalid] = chosen

    # ========= Learn CONTEXT MAPS (for UNKNOWN) =========
    known = tmp[tmp[fuel_col] != "UNKNOWN"]

    # (Brand, Model, Transmission)
    ctx_bmt = {}
    if all(c in known.columns for c in [brand_col, model_col, transm_col]):
        for key, series in known.groupby([brand_col, model_col, transm_col])[fuel_col]:
            m = series.mode()
            if not m.empty:
                ctx_bmt[key] = m.iloc[0]

    # (Brand, Model)
    ctx_bm = {}
    if all(c in known.columns for c in [brand_col, model_col]):
        for key, series in known.groupby([brand_col, model_col])[fuel_col]:
            m = series.mode()
            if not m.empty:
                ctx_bm[key] = m.iloc[0]

    # Brand only
    ctx_b = {}
    if brand_col in known.columns:
        for b, series in known.groupby(brand_col)[fuel_col]:
            m = series.mode()
            if not m.empty:
                ctx_b[b] = m.iloc[0]

    state = {
        "fuel_col": fuel_col,
        "brand_col": brand_col,
        "model_col": model_col,
        "transm_col": transm_col,
        "valid_set": dynamic_valids,
        "invalid_to_valid": invalid_to_valid,
        "ctx_bmt": ctx_bmt,
        "ctx_bm": ctx_bm,
        "ctx_b": ctx_b,
    }

    return state

In [32]:
def transform_fueltype_resolver(
    df: pd.DataFrame,
    state: Dict[str, Any]
) -> tuple[pd.DataFrame, dict, list]:
    """
    Transform step for fuel type correction using the statistics
    learned in `fit_fueltype_resolver`.

    Rules for each row:
    ------------------------------------------------------
    A) If fuel is already valid → keep as is.
    B) If fuel == 'UNKNOWN':
        1) try (Brand, Model, Transmission)
        2) try (Brand, Model)
        3) try Brand
        4) else remain UNKNOWN
    C) If fuel is invalid and appeared in training:
        - replace via invalid_to_valid[invalid]
    D) If fuel is unseen invalid:
        - try substring match inside the valid_set
        - else keep as is.

    Returns:
    - df_out : dataframe with corrected fuel types
    - corrections : log of corrections
    - still_invalid : fuel types that remain outside valid set & not UNKNOWN
    """
    df_out = df.copy()

    fuel_col   = state["fuel_col"]
    brand_col  = state["brand_col"]
    model_col  = state["model_col"]
    transm_col = state["transm_col"]

    valid_set        = state["valid_set"]
    invalid_to_valid = state["invalid_to_valid"]
    ctx_bmt          = state["ctx_bmt"]
    ctx_bm           = state["ctx_bm"]
    ctx_b            = state["ctx_b"]

    corrections = {}

    # Ensure normalization (safety)
    df_out[fuel_col] = (
        df_out[fuel_col]
        .fillna("UNKNOWN")
        .astype(str)
        .str.strip()
        .str.upper()
    )
    df_out[brand_col] = df_out[brand_col].astype(str).str.strip().str.upper()
    df_out[model_col] = df_out[model_col].astype(str).str.strip().str.upper()
    df_out[transm_col] = df_out[transm_col].astype(str).str.strip().str.upper()

    for idx, row in df_out.iterrows():
        original = row[fuel_col]

        # A) Already valid → skip
        if original in valid_set:
            continue

        # ---------- B) UNKNOWN fuel type ----------
        if original == "UNKNOWN":
            key_bmt = (row[brand_col], row[model_col], row[transm_col])
            key_bm  = (row[brand_col], row[model_col])
            key_b   = row[brand_col]

            if key_bmt in ctx_bmt:
                corrected = ctx_bmt[key_bmt]
            elif key_bm in ctx_bm:
                corrected = ctx_bm[key_bm]
            elif key_b in ctx_b:
                corrected = ctx_b[key_b]
            else:
                corrected = "UNKNOWN"

            df_out.at[idx, fuel_col] = corrected
            corrections[(idx, original)] = corrected
            continue

        # ---------- C) Invalid fuel known from training ----------
        if original in invalid_to_valid:
            corrected = invalid_to_valid[original]
            df_out.at[idx, fuel_col] = corrected
            corrections[(idx, original)] = corrected
            continue

        # ---------- D) Unseen invalid: attempt substring match ----------
        invalid_upper = str(original).upper()
        matches = [f for f in valid_set if invalid_upper in f]

        if len(matches) == 1:
            corrected = matches[0]
        elif len(matches) > 1:
            # If multiple: choose the most common among TRAIN-known valid_set
            # Here we fallback to the first match (stable, deterministic)
            corrected = matches[0]
        else:
            corrected = original  # keep as is

        df_out.at[idx, fuel_col] = corrected
        corrections[(idx, original)] = corrected

    # Identify still-invalid fuel types
    still_invalid = [
        f for f in df_out[fuel_col].unique()
        if f not in valid_set and f != "UNKNOWN"
    ]

    return df_out, corrections, still_invalid


### TRansmission

In [33]:
def fit_transmission_resolver(
    train_df: pd.DataFrame,
    valid_transmissions: list[str],
    transm_col: str = "transmission",
    brand_col: str = "Brand",
    model_col: str = "model",
    fuel_col: str = "fuelType",
) -> Dict[str, Any]:
    """
    Fit step for resolving invalid/ambiguous transmission values.

    Learns from TRAIN ONLY:
    1. Set of valid transmissions (starting from valid_transmissions,
       possibly expanded with new entries).
    2. Mapping invalid_transmission -> corrected_transmission for
       invalid strings seen in training (excluding 'UNKNOWN').
    3. Context maps for resolving 'UNKNOWN':
       - (Brand, Model, Fuel) -> mode(transmission)
       - (Brand, Model) -> mode(transmission)
       - Brand -> mode(transmission)

    Assumes basic string cleaning (upper/strip, NaN -> 'UNKNOWN') was applied
    earlier, but adds a small safety normalization.
    """
    tmp = train_df.copy()

    # Safety normalization
    tmp[transm_col] = (
        tmp[transm_col]
        .fillna("UNKNOWN")
        .astype(str)
        .str.strip()
        .str.upper()
    )
    tmp[brand_col] = tmp[brand_col].astype(str).str.strip().str.upper()
    tmp[model_col] = tmp[model_col].astype(str).str.strip().str.upper()
    tmp[fuel_col] = tmp[fuel_col].astype(str).str.strip().str.upper()

    valid_transmissions = [t.upper().strip() for t in valid_transmissions]
    valid_set = set(valid_transmissions)

    # Invalid transmissions observed in training
    invalid_trans = sorted(
        [t for t in tmp[transm_col].unique() if t not in valid_set],
        key=len,
        reverse=True,
    )

    invalid_to_valid: Dict[str, str] = {}

    # Learn mapping invalid_transmission -> corrected_transmission
    for invalid in invalid_trans:
        if invalid == "UNKNOWN":
            continue  # UNKNOWN handled by context, not here

        subset = tmp[tmp[transm_col] == invalid]
        if subset.empty:
            continue

        # Substring match against known valid transmissions
        matches = [v for v in valid_transmissions if invalid in v]

        if len(matches) == 1:
            # Unique match -> typo fix
            chosen = matches[0]

        elif len(matches) == 0:
            # No match -> promote as new valid transmission
            chosen = invalid
            valid_set.add(invalid)
            valid_transmissions.append(invalid)

        else:
            # Multiple matches -> pick most frequent among those matches in TRAIN
            df_f = tmp[tmp[transm_col].isin(matches)]
            if not df_f.empty:
                chosen = df_f[transm_col].mode().iloc[0]
            else:
                chosen = matches[0]

        invalid_to_valid[invalid] = chosen

    # Context maps for UNKNOWN inference (only rows with known transmission)
    known = tmp[tmp[transm_col] != "UNKNOWN"]

    ctx_bmf = {}  # (Brand, Model, Fuel) -> transmission
    if all(c in known.columns for c in [brand_col, model_col, fuel_col]):
        for key, series in known.groupby([brand_col, model_col, fuel_col])[transm_col]:
            m = series.mode()
            if not m.empty:
                ctx_bmf[key] = m.iloc[0]

    ctx_bm = {}  # (Brand, Model) -> transmission
    if all(c in known.columns for c in [brand_col, model_col]):
        for key, series in known.groupby([brand_col, model_col])[transm_col]:
            m = series.mode()
            if not m.empty:
                ctx_bm[key] = m.iloc[0]

    ctx_b = {}   # Brand -> transmission
    if brand_col in known.columns:
        for b, series in known.groupby(brand_col)[transm_col]:
            m = series.mode()
            if not m.empty:
                ctx_b[b] = m.iloc[0]

    # Global most frequent valid transmission (for strong fallback)
    trans_freq = known[transm_col].value_counts()
    global_most_common = trans_freq.index[0] if not trans_freq.empty else None

    state = {
        "transm_col": transm_col,
        "brand_col": brand_col,
        "model_col": model_col,
        "fuel_col": fuel_col,
        "valid_set": valid_set,
        "invalid_to_valid": invalid_to_valid,
        "ctx_bmf": ctx_bmf,
        "ctx_bm": ctx_bm,
        "ctx_b": ctx_b,
        "global_most_common": global_most_common,
    }

    return state


In [34]:
def transform_transmission_resolver(
    df: pd.DataFrame,
    state: Dict[str, Any],
) -> tuple[pd.DataFrame, dict, list]:
    """
    Transform step for transmission correction using the statistics
    learned in `fit_transmission_resolver`.

    For each row:

    A) If transmission is already valid -> keep as is.
    B) If transmission == 'UNKNOWN':
       - If (Brand, Model, Fuel) all 'UNKNOWN' -> keep 'UNKNOWN'.
       - Else try context maps in order:
         1) (Brand, Model, Fuel)
         2) (Brand, Model)
         3) Brand
         4) global most common transmission
    C) If transmission is invalid and appeared in training:
       - replace with invalid_to_valid[invalid].
    D) If transmission is unseen invalid:
       - try substring match against valid_set;
       - if several matches, take the first (deterministic);
       - if no matches, keep original.

    Returns
    -------
    df_out : pd.DataFrame
        Copy of df with corrected transmissions.
    corrections : dict
        (row_index, original_transmission) -> corrected_transmission.
    still_invalid : list
        Transmission values not in valid_set and not 'UNKNOWN'.
    """
    df_out = df.copy()

    transm_col = state["transm_col"]
    brand_col  = state["brand_col"]
    model_col  = state["model_col"]
    fuel_col   = state["fuel_col"]

    valid_set        = state["valid_set"]
    invalid_to_valid = state["invalid_to_valid"]
    ctx_bmf          = state["ctx_bmf"]
    ctx_bm           = state["ctx_bm"]
    ctx_b            = state["ctx_b"]
    global_most      = state["global_most_common"]

    corrections = {}

    # Safety normalization
    df_out[transm_col] = (
        df_out[transm_col]
        .fillna("UNKNOWN")
        .astype(str)
        .str.strip()
        .str.upper()
    )
    df_out[brand_col] = df_out[brand_col].astype(str).str.strip().str.upper()
    df_out[model_col] = df_out[model_col].astype(str).str.strip().str.upper()
    df_out[fuel_col]  = df_out[fuel_col].astype(str).str.strip().str.upper()

    for idx, row in df_out.iterrows():
        original = row[transm_col]

        # A) Already valid
        if original in valid_set:
            continue

        # B) UNKNOWN transmissions
        if original == "UNKNOWN":
            brand = row[brand_col]
            model = row[model_col]
            fuel  = row[fuel_col]

            # If there is literally no context, keep UNKNOWN
            if brand == "UNKNOWN" and model == "UNKNOWN" and fuel == "UNKNOWN":
                corrected = "UNKNOWN"
            else:
                key_bmf = (brand, model, fuel)
                key_bm  = (brand, model)
                key_b   = brand

                if key_bmf in ctx_bmf:
                    corrected = ctx_bmf[key_bmf]
                elif key_bm in ctx_bm:
                    corrected = ctx_bm[key_bm]
                elif key_b in ctx_b:
                    corrected = ctx_b[key_b]
                else:
                    corrected = global_most if global_most is not None else "UNKNOWN"

            df_out.at[idx, transm_col] = corrected
            corrections[(idx, original)] = corrected
            continue

        # C) Invalid seen in training
        if original in invalid_to_valid:
            corrected = invalid_to_valid[original]
            df_out.at[idx, transm_col] = corrected
            corrections[(idx, original)] = corrected
            continue

        # D) Invalid unseen in training: substring-based guess
        invalid_upper = str(original).upper()
        matches = [t for t in valid_set if invalid_upper in t]

        if len(matches) == 1:
            corrected = matches[0]
        elif len(matches) > 1:
            corrected = matches[0]  # deterministic fallback
        else:
            corrected = original    # keep as is

        df_out.at[idx, transm_col] = corrected
        corrections[(idx, original)] = corrected

    still_invalid = [
        t for t in df_out[transm_col].unique()
        if t not in valid_set and t != "UNKNOWN"
    ]

    return df_out, corrections, still_invalid


## General pre processing apllied

We will now apply what we previously established 

So: 
Categorial variables will all receive nan replacement with Unknown and basic string normalizer. 
Define valid brands and models, etc
invalid brands replacer

In [42]:
full_train_dataset = full_train_dataset.drop(columns=['carID', 'hasDamage'])

In [43]:
full_train_dataset.head()

,Brand,model,year,price,transmission,mileage,fuelType,tax,mpg,engineSize,paintQuality%,previousOwners
0,VW,Golf,2016.0,22290,Semi-Auto,28421.0,Petrol,NaN,11.417268,2.0,63.0,4.000000
1,Toyota,Yaris,2019.0,13790,Manual,4589.0,Petrol,145.0,47.900000,1.5,50.0,1.000000
2,Audi,Q2,2019.0,24990,Semi-Auto,3624.0,Petrol,145.0,40.900000,1.5,56.0,4.000000
3,Ford,FIESTA,2018.0,12500,anual,9102.0,Petrol,145.0,65.700000,1.0,50.0,-2.340306
4,BMW,2 Series,2019.0,22995,Manual,1000.0,Petrol,145.0,42.800000,1.5,97.0,3.000000


In [45]:
# categorical features 
cat_feat = ['Brand', 'model', 'transmission', 'fuelType']

# numerical features 
num_feat = ['year', 'mileage', 'engineSize', 'tax', 'mpg', 'paintQuality%', 'previousOwners']


In [46]:
def preprocess_categorical(df, columns, remove_middle_spaces=True, allow_extra_chars=""):
    df = df.copy()

    for col in columns:
        # Step 1: replace NaN with "UNKNOWN"
        df[col] = fill_unknown(df[col])

        # Step 2: apply basic string transformer
        df[col] = df[col].apply(
            lambda x: basic_string_transformer(
                x,
                remove_middle_spaces=remove_middle_spaces,
                allow_extra_chars=allow_extra_chars
            )
        )

        # Optional: ensure dtype string
        df[col] = df[col].astype("string")

    return df


In [ ]:
full_train_dataset = preprocess_categorical(
    full_train_dataset,
    cat_feat,
    remove_middle_spaces=True,
    allow_extra_chars=""
)

In [ ]:
# Brand deterministic preprocessing 

invalids = sorted(
    [b for b in full_train_dataset['Brand'].unique() if b not in valid_brands],
    key=len
)

full_train_dataset, corrections, remaining_invalids = correct_invalid_brands_in_df(
    full_train_dataset,
    col='Brand',
    valid_brands=valid_brands,
    invalids=invalids
)